# AI Pest Finder — Dataset Setup and Final Model

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf


In [3]:
DATASET_PATH = Path.home() / "Downloads" / "ip102_v1.1"

print("Dataset path:", DATASET_PATH)
print("Dataset exists:", DATASET_PATH.exists())


Dataset path: C:\Users\jeeva\Downloads\ip102_v1.1
Dataset exists: True


In [4]:
selected_classes = {
    0: "rice leaf roller",
    3: "asiatic rice borer",
    7: "brown plant hopper",
    11: "rice leafhopper",
    18: "black cutworm",
    22: "corn borer",
    23: "army worm",
    24: "aphids",
    54: "Thrips",
    71: "Trialeurodes vaporariorum"
}

NUM_CLASSES = 10
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32


In [5]:
def read_ip102_split(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            image_name, class_id = line.split()
            records.append({
                "image": image_name,
                "class_id": int(class_id)
            })

    return pd.DataFrame(records)


In [6]:
train_df = read_ip102_split(DATASET_PATH / "train.txt")
val_df = read_ip102_split(DATASET_PATH / "val.txt")
test_df = read_ip102_split(DATASET_PATH / "test.txt")

train_df["class_name"] = train_df["class_id"].map(selected_classes)
val_df["class_name"] = val_df["class_id"].map(selected_classes)
test_df["class_name"] = test_df["class_id"].map(selected_classes)

train_selected = train_df[
    train_df["class_id"].isin(selected_classes.keys())
].copy()

val_selected = val_df[
    val_df["class_id"].isin(selected_classes.keys())
].copy()

test_selected = test_df[
    test_df["class_id"].isin(selected_classes.keys())
].copy()

print("Train:", len(train_selected))
print("Validation:", len(val_selected))
print("Test:", len(test_selected))


Train: 7612
Validation: 1268
Test: 3814


In [7]:
class_to_label = {
    class_id: label
    for label, class_id in enumerate(selected_classes.keys())
}

label_to_class = {
    label: selected_classes[class_id]
    for class_id, label in class_to_label.items()
}

train_selected["label"] = train_selected["class_id"].map(class_to_label)
val_selected["label"] = val_selected["class_id"].map(class_to_label)
test_selected["label"] = test_selected["class_id"].map(class_to_label)


In [8]:
def create_dataset(df, shuffle=False):
    image_paths = [
        str(DATASET_PATH / "images" / image_name)
        for image_name in df["image"]
    ]

    labels = df["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(
            image,
            [IMG_HEIGHT, IMG_WIDTH]
        )
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.cache()

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(df), 10000),
            reshuffle_each_iteration=True
        )

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_dataset = create_dataset(train_selected, shuffle=True)
val_dataset = create_dataset(val_selected)
test_dataset = create_dataset(test_selected)

print("Datasets created successfully.")


Datasets created successfully.


## MobileNetV2

In [10]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False


In [11]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    tf.keras.layers.Rescaling(scale=2.0, offset=-1.0),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [12]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 188s 605ms/step - accuracy: 0.5259 - loss: 1.4748 - val_accuracy: 0.6546 - val_loss: 1.0500 - learning_rate: 0.0010
Epoch 2/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 187s 662ms/step - accuracy: 0.6653 - loss: 1.0051 - val_accuracy: 0.6719 - val_loss: 0.9675 - learning_rate: 0.0010
Epoch 3/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 133s 559ms/step - accuracy: 0.7236 - loss: 0.8434 - val_accuracy: 0.7106 - val_loss: 0.8700 - learning_rate: 0.0010
Epoch 4/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 125s 488ms/step - accuracy: 0.7541 - loss: 0.7388 - val_accuracy: 0.7256 - val_loss: 0.8523 - learning_rate: 0.0010
Epoch 5/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 118s 497ms/step - accuracy: 0.7910 - loss: 0.6299 - val_accuracy: 0.7279 - val_loss: 0.8586 - learning_rate: 0.0010
Epoch 6/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 135s 467ms/step - accuracy: 0.8075 - loss: 0.5622 - val_accuracy: 0.7334 - val_loss: 0.8410 - learning_rate: 0.0010
Epoch 7/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 119s 501ms/step - accura

In [13]:
test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


120/120 ━━━━━━━━━━━━━━━━━━━━ 54s 445ms/step - accuracy: 0.7205 - loss: 0.8592
Test Accuracy: 72.05%


## Fine-tuning

In [15]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [16]:
fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7
    )
]

history_finetune = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    callbacks=fine_tune_callbacks
)


Epoch 1/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 148s 552ms/step - accuracy: 0.8527 - loss: 0.4412 - val_accuracy: 0.7461 - val_loss: 0.8283 - learning_rate: 1.0000e-05
Epoch 2/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 132s 511ms/step - accuracy: 0.8732 - loss: 0.3871 - val_accuracy: 0.7508 - val_loss: 0.8315 - learning_rate: 1.0000e-05
Epoch 3/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 145s 524ms/step - accuracy: 0.8861 - loss: 0.3467 - val_accuracy: 0.7571 - val_loss: 0.8200 - learning_rate: 1.0000e-05
Epoch 4/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 130s 545ms/step - accuracy: 0.8981 - loss: 0.3174 - val_accuracy: 0.7603 - val_loss: 0.8261 - learning_rate: 1.0000e-05
Epoch 5/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 127s 534ms/step - accuracy: 0.9094 - loss: 0.2825 - val_accuracy: 0.7666 - val_loss: 0.8241 - learning_rate: 1.0000e-05


In [17]:
test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")


120/120 ━━━━━━━━━━━━━━━━━━━━ 45s 371ms/step - accuracy: 0.7465 - loss: 0.8414
Final Test Accuracy: 74.65%


## Final model and class mapping

In [19]:
final_model_path = "final_pest_mobilenetv2_74_46.keras"
model.save(final_model_path)

class_mapping = {
    str(i): label_to_class[i]
    for i in range(NUM_CLASSES)
}

with open("pest_class_mapping.json", "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, indent=4)

print("Final model saved:", final_model_path)
print("Class mapping saved: pest_class_mapping.json")


Final model saved: final_pest_mobilenetv2_74_46.keras
Class mapping saved: pest_class_mapping.json
